# Assignment Instructions - Part 2: Classical Machine Learning

Statistics **describe**. Machine learning **predicts**.

In Part 1 you summarized observations without fitting a predictive model. In this notebook you will use [scikit-learn](https://scikit-learn.org/stable/) to ask a harder question: *given what I have measured, can I predict something I have not measured, and how would I know if I was fooling myself?*

You are expected to look things up. The scikit-learn [user guide](https://scikit-learn.org/stable/user_guide.html) and API reference are the primary sources. This notebook supplies a working scaffold, some vocabulary, and questions; it intentionally does not explain every function.

## Learning goals

By the end of this activity, you should be able to:

- distinguish supervised from unsupervised learning, and regression from classification;
- build a train/test evaluation that does not leak information and compare against a baseline; and
- explain, with evidence from your own experiments, how dataset size and the number of features limit what a model can learn (overfitting, the curse of dimensionality, and the "rule of 10").

This notebook ends by counting the choices you made by hand. Part 3 will ask a tool (TPOT) to make those choices for you.

## 1. Setup

Scikit-learn is new to this repository, so make sure your environment has it. If the import below fails, update the environment as described in the README (`environment.yml` now lists `scikit-learn`).

In [1]:
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import sklearn

from sklearn.datasets import load_wine
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

RANDOM_SEED = 802
rng = np.random.default_rng(RANDOM_SEED)

print(f"Python: {sys.version.split()[0]}")
print(f"NumPy: {np.__version__}")
print(f"pandas: {pd.__version__}")
print(f"scikit-learn: {sklearn.__version__}")

ModuleNotFoundError: No module named 'sklearn'

## 2. One dataset, three questions

We will use the classic **wine** dataset that ships with scikit-learn: chemical measurements of wines grown by three different cultivars (grape varieties). It is small, real, and well-behaved, which makes it a good place to see what the methods do before a messy research dataset hides the lesson.

Vocabulary to look up if it is unfamiliar: *feature*, *target/label*, *supervised*, *unsupervised*.

With the same table we can ask three different kinds of question:

| Question | Type | Target |
|---|---|---|
| Which cultivar made this wine? | classification | `cultivar` (category) |
| What is this wine's alcohol content? | regression | `alcohol` (continuous number) |
| Do the wines fall into natural groups? | clustering | none (unsupervised) |

In [ ]:
wine = load_wine(as_frame=True)
data = wine.frame.rename(columns={"target": "cultivar"})

print(f"{data.shape[0]} observations, {data.shape[1] - 1} measured features + 1 label")
data.head()

Before modeling, inspect the data as you did in Part 1. Use whatever you need from `pandas` to answer these, and write your answers in a markdown cell:

- Are there missing values? What are the feature scales, and does that matter for any model you know?
- Are the three cultivars equally common? Why might that matter for choosing an evaluation metric?

In [ ]:
# Your inspection here.

## 3. Training data, testing data, and a baseline

A model that is scored on the data it learned from tells you how well it *memorized*. To estimate how it will do on new observations, hold some data back. Two habits matter more than any particular algorithm:

1. **Split before you do anything that learns from the data** (scaling, feature selection, fitting). Look up *data leakage*. Putting preprocessing inside a `Pipeline` is the standard defense.
2. **Compare against a baseline.** A score is meaningless until you know what a trivial model achieves. See `sklearn.dummy`.

### Classification

The cell below is a complete worked example. Read it line by line and look up anything you cannot explain (`stratify`, `random_state`, `make_pipeline`).

In [ ]:
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression

X = data.drop(columns="cultivar")
y = data["cultivar"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, stratify=y, random_state=RANDOM_SEED
)

baseline = DummyClassifier(strategy="most_frequent").fit(X_train, y_train)
clf = make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000))
clf.fit(X_train, y_train)

print(f"Baseline accuracy: {baseline.score(X_test, y_test):.2f}")
print(f"Model train accuracy: {clf.score(X_train, y_train):.2f}")
print(f"Model test accuracy:  {clf.score(X_test, y_test):.2f}")

**Your turn.**

1. This test set has only about 45 wines. Change `random_state` several times. How much does the test accuracy move? What does that say about trusting a single split? (Look at `cross_val_score` and `RepeatedStratifiedKFold`.)
2. Accuracy hides *which* mistakes are made. Look up `confusion_matrix` or `classification_report` and inspect the errors. Would any of them matter more than others in a scientific setting?
3. Swap `LogisticRegression` for two other classifiers of your choice (for example `KNeighborsClassifier`, `RandomForestClassifier`, `SVC`). Do they differ by more than the split-to-split noise you measured in step 1?

In [ ]:
# Your classification experiments here.

### Regression

Now predict a continuous quantity. We will predict `alcohol` from the other chemical measurements. The cultivar label is dropped from the features. Think about why: would you have it available when you wanted to use this model?

In [ ]:
from sklearn.dummy import DummyRegressor
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_absolute_error, r2_score

X_reg = data.drop(columns=["alcohol", "cultivar"])
y_reg = data["alcohol"]

Xr_train, Xr_test, yr_train, yr_test = train_test_split(
    X_reg, y_reg, test_size=0.25, random_state=RANDOM_SEED
)

reg = make_pipeline(StandardScaler(), Ridge(alpha=1.0)).fit(Xr_train, yr_train)
reg_baseline = DummyRegressor(strategy="mean").fit(Xr_train, yr_train)

for name, model in [("baseline (mean)", reg_baseline), ("ridge", reg)]:
    pred = model.predict(Xr_test)
    print(f"{name:16s} R2 = {r2_score(yr_test, pred):6.2f}   MAE = {mean_absolute_error(yr_test, pred):.2f}")

**Your turn.**

1. Plot predicted versus observed `alcohol` for the test set. Where does the model do poorly?
2. What does an R² near 0, or a *negative* R², mean? Why can the test R² be negative?
3. Look up what `alpha` does in `Ridge`. Try several values. Is the change larger than the noise from the choice of split?
4. Part 1 used correlation to describe linear association. What does a regression model give you that a correlation coefficient does not, and what does it still not give you?

In [ ]:
# Your regression experiments here.

### Clustering (no labels)

Clustering is **unsupervised**: the algorithm never sees `cultivar`. We keep the labels only to check afterwards whether the structure it found means anything. In real research you often do not have such labels, which is exactly why clustering results need skepticism.

In [ ]:
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.metrics import adjusted_rand_score

X_scaled = StandardScaler().fit_transform(X)
clusters = KMeans(n_clusters=3, n_init=10, random_state=RANDOM_SEED).fit_predict(X_scaled)
print(f"Adjusted Rand index vs. cultivar: {adjusted_rand_score(y, clusters):.2f}")

coords = PCA(n_components=2).fit_transform(X_scaled)
fig, axes = plt.subplots(1, 2, figsize=(11, 4.5), sharex=True, sharey=True)
axes[0].scatter(coords[:, 0], coords[:, 1], c=y, cmap="viridis", alpha=0.8)
axes[0].set_title("Colored by true cultivar")
axes[1].scatter(coords[:, 0], coords[:, 1], c=clusters, cmap="viridis", alpha=0.8)
axes[1].set_title("Colored by KMeans cluster")
for ax in axes:
    ax.set_xlabel("PC 1")
axes[0].set_ylabel("PC 2")
plt.show()

**Your turn.**

1. We told KMeans that `n_clusters=3` because we already knew the answer. Without that knowledge, how would you choose the number of clusters? (Look up the silhouette score and the "elbow" heuristic, and test them here.)
2. Remove the `StandardScaler` and rerun. What changes, and why?
3. What is PCA doing in the plots, and what can be lost when you project 13 features onto 2?
4. Which of the three questions (classification, regression, clustering) did you find easiest to evaluate honestly, and why?

In [ ]:
# Your clustering experiments here.

## 4. Overfitting

Flexible models can fit noise as well as signal. The signature is a growing gap between training and testing performance. A decision tree is a convenient place to see it because its flexibility is controlled by one number: `max_depth`.

The cell below uses `validation_curve`, which repeats cross-validation across values of one hyperparameter. Run it, then answer the questions.

In [ ]:
from sklearn.model_selection import validation_curve
from sklearn.tree import DecisionTreeRegressor

depths = np.arange(1, 13)
train_scores, val_scores = validation_curve(
    DecisionTreeRegressor(random_state=RANDOM_SEED),
    X_reg,
    y_reg,
    param_name="max_depth",
    param_range=depths,
    cv=5,
    scoring="r2",
)

plt.figure(figsize=(7, 4.5))
plt.plot(depths, train_scores.mean(axis=1), "o-", label="train")
plt.plot(depths, val_scores.mean(axis=1), "s-", label="validation")
plt.xlabel("max_depth")
plt.ylabel("R$^2$")
plt.legend()
plt.grid(alpha=0.25)
plt.show()

- At what depth does the model start to overfit? How would you describe that point to a collaborator who has not taken a machine learning course?
- Many research datasets have noisy measurements. How would you expect this curve to change if you added more noise to `y_reg`? Try it.
- Why is it a problem to pick the depth that scores best on the *test* set and then report that test score?

## 5. Dataset size and the curse of dimensionality

This is the central idea of the day, and it is common in science: you often measure *many* properties of *few* samples (spectra, genomes, sensor arrays, simulation parameters).

**The rule of 10** is an informal guideline: have roughly **10 observations per feature** (some versions say per model parameter or per class-feature). It is not a theorem, and for some problems it is too lax or too strict. It is still a useful first sanity check before you trust any model. Look up where the rule comes from and what its caveats are (a good search phrase is "events per variable").

The **curse of dimensionality** is the broader phenomenon behind it: as the number of features grows, the volume of the feature space grows so quickly that a fixed number of observations covers it ever more sparsely. Distances between points become less informative, and models find spurious patterns that do not generalize.

### 5a. How far is "near" in high dimensions?

Many methods (k-nearest neighbors, clustering, kernels) rely on distance. Run this experiment, then explain the result in your own words.

In [ ]:
from scipy.spatial.distance import cdist

n_points = 500
rows = []
for n_dims in [1, 2, 5, 10, 50, 100, 500, 1000]:
    points = rng.uniform(size=(n_points, n_dims))
    dists = cdist(points[:1], points[1:])[0]
    rows.append(
        {
            "dimensions": n_dims,
            "nearest": dists.min(),
            "farthest": dists.max(),
            "relative_contrast": (dists.max() - dists.min()) / dists.min(),
        }
    )

pd.DataFrame(rows).round(2)

- What happens to `relative_contrast` as dimensions increase? What does this imply for a "nearest neighbor" method?
- How many random points would you need to keep the same *density* of coverage in 10 dimensions as 100 points give in 1? (Do the arithmetic.)

### 5b. Add useless features and watch the model

The wine data has 13 features and 178 observations, about 14 per feature, so it passes the rule of 10 (barely). We will add columns of pure random noise that carry no information and see how performance responds. Because we *know* the extra features are useless, any drop in performance is caused by dimensionality alone.

The helper below is provided. Read it before you use it.

In [ ]:
def score_with_noise_features(model, X, y, n_noise, n_repeats=5, seed=RANDOM_SEED):
    """Return mean and std of CV accuracy after appending `n_noise` random columns."""
    local_rng = np.random.default_rng(seed)
    scores = []
    for _ in range(n_repeats):
        noise = local_rng.normal(size=(len(X), n_noise))
        X_aug = np.hstack([np.asarray(X), noise])
        scores.extend(cross_val_score(model, X_aug, y, cv=5))
    return np.mean(scores), np.std(scores)


from sklearn.neighbors import KNeighborsClassifier

knn = make_pipeline(StandardScaler(), KNeighborsClassifier())
noise_counts = [0, 10, 50, 100, 200, 500]

rows = []
for k in noise_counts:
    mean_acc, std_acc = score_with_noise_features(knn, X, y, k)
    total_features = X.shape[1] + k
    rows.append(
        {
            "noise_features": k,
            "total_features": total_features,
            "obs_per_feature": len(X) / total_features,
            "cv_accuracy": mean_acc,
            "cv_std": std_acc,
        }
    )

noise_results = pd.DataFrame(rows)
noise_results.round(3)

**Your turn.**

1. Plot `cv_accuracy` against `obs_per_feature` (a log axis may help). Mark where the rule of 10 would say you have run out of data. Does performance degrade at about that point, earlier, or later?
2. Repeat the experiment with `LogisticRegression`, `RandomForestClassifier`, and at least one other model. Which models are most and least sensitive to useless features? Offer a hypothesis for why.
3. Rerun with a *smaller* subset of the real observations (for example 60 wines). How does the picture shift? What combination of observations and features do you need for the model to be trustworthy?
4. Use `SelectKBest` or a model with L1 regularization to try to *recover* from the noise features. Remember the leakage warning from Section 3: where in the pipeline must the selection step go, and what happens to your score if you put it outside?
5. Write down a rule you would give a collaborator, in your own words, for deciding if their dataset is large enough. Include at least one situation where the rule of 10 would mislead you.

In [ ]:
# Your noise-feature experiments here.

## 6. Reproducibility and handoff

Machine learning adds new sources of irreproducibility on top of the ones you already manage: random splits, random initialization, library versions, and silent changes in defaults. Make sure your notebook would let a teammate reproduce your numbers.

- Did every random step (splits, models, noise generation) receive a seed? Where did you rely on a default?
- Are your final conclusions based on a single split, or on repeated cross-validation?
- Record the library versions printed in Section 1 alongside your results.

## 7. Bridge to AutoML: count your choices

Think back over everything you decided in this notebook: the preprocessing, the model family, its hyperparameters, whether to select features, the evaluation split, the metric. Each choice affects the result, and the combinations multiply.

The cell below searches a *tiny* slice of that space by brute force with `GridSearchCV`. It uses a Pipeline whose steps themselves can be swapped, which is the same abstraction that AutoML tools build on.

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import GridSearchCV
from sklearn.pipeline import Pipeline
from sklearn.svm import SVC

pipe = Pipeline([("scale", StandardScaler()), ("model", LogisticRegression(max_iter=1000))])

param_grid = [
    {"model": [LogisticRegression(max_iter=1000)], "model__C": [0.01, 0.1, 1, 10]},
    {"model": [SVC()], "model__C": [0.1, 1, 10], "model__kernel": ["linear", "rbf"]},
    {"model": [RandomForestClassifier(random_state=RANDOM_SEED)], "model__n_estimators": [50, 200]},
]

search = GridSearchCV(pipe, param_grid, cv=5)
search.fit(X_train, y_train)

print(f"Candidate pipelines evaluated: {len(search.cv_results_['params'])}")
print(f"Best CV score: {search.best_score_:.3f}")
print(f"Held-out test score: {search.score(X_test, y_test):.3f}")
print(f"Best settings: {search.best_params_}")

**Your turn.**

1. How many model fits did that search require in total? (Count candidates times cross-validation folds.) Now imagine adding 5 preprocessing options, 10 models, and 5 hyperparameters each with 10 values. What happens to the cost?
2. Look at the top several rows of `pd.DataFrame(search.cv_results_)`. Are the best candidates meaningfully different from each other, given the standard deviation across folds?
3. The best CV score is *selected* from many candidates. Why is it therefore an optimistically biased estimate, and why do we keep a separate test set that the search never sees? Connect this to what you saw in the overfitting section.
4. Combine this with the curse of dimensionality: what could go wrong if you ran a very large automated search on a dataset that fails the rule of 10?

Part 3 will use **TPOT**, which searches over pipelines like this one using genetic programming. If you want to get ahead, read the [TPOT documentation](https://epistasislab.github.io/tpot/) and note what it asks you to specify and what it decides for you.

## Reflection

In a short response, address all three prompts:

- Was the dataset large enough for each question you asked of it? Give evidence from your experiments, not an opinion.
- Was machine learning necessary here? What would a simpler model, a Part 1 summary, or domain knowledge have given you?
- In your own research, how many features and observations do you have? Would you trust a model built on them, and what would you check first?

## Before the next class

Keep this notebook. We will automate the model selection you did by hand. Come with one dataset from your own work (or a public one) described in a few sentences: its number of observations, number of features, and the prediction question you would ask of it.